# Notebook 08 — GraphSAGE Explainability

## Dynamic Heterogeneous Graph Neural Network for Bank Marketing Prediction

### Objective

Explain predictions made by the **final frozen heterogeneous GraphSAGE model**.

This notebook does **not** retrain, tune, or modify the final model.

Because the model is a neural heterogeneous graph model, this notebook uses a graph-native explanation approach:

**PyTorch Geometric `GNNExplainer` through the `Explainer` API**

The explanations focus on:

- customer-node feature importance
- neighboring graph relationships
- edge/relation importance
- individual customer prediction explanations

### Important

The final model remains:

```text
artifacts/models/hetero_graphsage_final.pt
```

No weights are changed.

We explain representative test-set predictions:

- True Positive
- False Positive
- False Negative

This gives both successful and failure-case explanations.


# 1. Required Artifacts

Notebook 08 requires the frozen model, graph, and final predictions from Notebooks 06–07.

Expected:

```text
artifacts/
├── graph/
│   └── bank_heterodata.pt
│
├── models/
│   └── hetero_graphsage_final.pt
│
└── results/
    └── final/
        ├── final_test_metrics.json
        └── final_predictions.csv
```

Error-analysis artifacts from Notebook 07 are also used when available.


In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.data import HeteroData
from torch_geometric.nn import HeteroConv, SAGEConv
from torch_geometric.explain import Explainer, GNNExplainer

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

GRAPH_PATH = PROJECT_ROOT / "artifacts" / "graph" / "bank_heterodata.pt"
FINAL_MODEL_PATH = PROJECT_ROOT / "artifacts" / "models" / "hetero_graphsage_final.pt"
FINAL_METRICS_PATH = PROJECT_ROOT / "artifacts" / "results" / "final" / "final_test_metrics.json"
FINAL_PREDICTIONS_PATH = PROJECT_ROOT / "artifacts" / "results" / "final" / "final_predictions.csv"

OUTPUT_DIR = PROJECT_ROOT / "artifacts" / "results" / "explainability"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXPLANATION_SUMMARY_PATH = OUTPUT_DIR / "explanation_summary.csv"
FEATURE_IMPORTANCE_PATH = OUTPUT_DIR / "feature_importance.csv"
RELATION_IMPORTANCE_PATH = OUTPUT_DIR / "relation_importance.csv"
PLOT_PATH = OUTPUT_DIR / "feature_importance.png"
CASE_PATH = OUTPUT_DIR / "representative_cases.json"

required = [
    GRAPH_PATH,
    FINAL_MODEL_PATH,
    FINAL_METRICS_PATH,
    FINAL_PREDICTIONS_PATH,
]

missing = [str(p) for p in required if not p.exists()]

if missing:
    raise FileNotFoundError(
        "Required artifact(s) missing:\n" + "\n".join(missing)
    )

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", DEVICE)
print("All required artifacts found.")


# 2. Load Frozen Graph, Model, and Final Predictions

The model checkpoint is loaded exactly as produced by Notebook 06.

No optimizer is created.


In [ ]:
try:
    data = torch.load(
        GRAPH_PATH,
        map_location="cpu",
        weights_only=False
    )
except TypeError:
    data = torch.load(
        GRAPH_PATH,
        map_location="cpu"
    )

try:
    checkpoint = torch.load(
        FINAL_MODEL_PATH,
        map_location="cpu",
        weights_only=False
    )
except TypeError:
    checkpoint = torch.load(
        FINAL_MODEL_PATH,
        map_location="cpu"
    )

with open(FINAL_METRICS_PATH, "r", encoding="utf-8") as f:
    final_metrics = json.load(f)

predictions = pd.read_csv(FINAL_PREDICTIONS_PATH)

assert isinstance(data, HeteroData)
assert checkpoint["freeze_status"] == "FINAL_FROZEN_MODEL"

required_prediction_columns = {
    "customer_index",
    "y_true",
    "predicted_probability",
    "y_pred",
}

assert required_prediction_columns.issubset(
    predictions.columns
)

data = data.to(DEVICE)

train_mask = data["customer"].train_mask.bool()
val_mask = data["customer"].val_mask.bool()
test_mask = data["customer"].test_mask.bool()

print("Frozen model status:", checkpoint["freeze_status"])
print("Model configuration:")
print(json.dumps(checkpoint["model_config"], indent=2))
print("Final test metrics:")
print(json.dumps(final_metrics, indent=2))


# 3. Reconstruct the Exact GraphSAGE Architecture

The architecture must match Notebook 06 exactly.

The loaded state dictionary is not changed.


In [ ]:
class HeteroGraphSAGE(nn.Module):
    def __init__(
        self,
        metadata,
        hidden_dim=128,
        dropout=0.20
    ):
        super().__init__()

        _, edge_types = metadata
        self.dropout = dropout

        self.conv1 = HeteroConv({
            edge_type: SAGEConv(
                (-1, -1),
                hidden_dim,
                aggr="mean"
            )
            for edge_type in edge_types
        }, aggr="sum")

        self.conv2 = HeteroConv({
            edge_type: SAGEConv(
                (-1, -1),
                hidden_dim,
                aggr="mean"
            )
            for edge_type in edge_types
        }, aggr="sum")

        self.classifier = nn.Linear(
            hidden_dim,
            1
        )

    def forward(
        self,
        x_dict,
        edge_index_dict
    ):
        x_dict = self.conv1(
            x_dict,
            edge_index_dict
        )

        x_dict = {
            key: F.dropout(
                F.relu(value),
                p=self.dropout,
                training=self.training
            )
            for key, value in x_dict.items()
        }

        x_dict = self.conv2(
            x_dict,
            edge_index_dict
        )

        x_dict = {
            key: F.dropout(
                F.relu(value),
                p=self.dropout,
                training=self.training
            )
            for key, value in x_dict.items()
        }

        return self.classifier(
            x_dict["customer"]
        ).squeeze(-1)


config = checkpoint["model_config"]

assert config["hidden_dim"] == 128
assert abs(config["dropout"] - 0.20) < 1e-12

model = HeteroGraphSAGE(
    data.metadata(),
    hidden_dim=config["hidden_dim"],
    dropout=config["dropout"]
).to(DEVICE)

model.eval()

with torch.no_grad():
    model(
        data.x_dict,
        data.edge_index_dict
    )

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()

print("Frozen GraphSAGE restored.")


# 4. Verify Frozen Model Reproducibility

Before explanation, verify that the restored model reproduces the final test probabilities.

This ensures that explanations are generated for the actual frozen model.


In [ ]:
with torch.no_grad():
    logits = model(
        data.x_dict,
        data.edge_index_dict
    )

test_indices = (
    torch.where(test_mask)[0]
    .detach()
    .cpu()
    .numpy()
)

test_probabilities = (
    torch.sigmoid(logits[test_mask])
    .detach()
    .cpu()
    .numpy()
)

saved_predictions = predictions.copy()

saved_predictions["customer_index"] = (
    saved_predictions["customer_index"].astype(int)
)

recomputed = pd.DataFrame({
    "customer_index": test_indices,
    "recomputed_probability": test_probabilities
})

check = saved_predictions.merge(
    recomputed,
    on="customer_index",
    how="inner",
    validate="one_to_one"
)

assert len(check) == len(saved_predictions)

max_difference = (
    check["predicted_probability"]
    - check["recomputed_probability"]
).abs().max()

print(
    "Maximum probability difference:",
    max_difference
)

assert max_difference < 1e-7

print("Frozen model reproducibility PASSED.")


# 5. Identify Representative Test Cases

We explain three representative test customers:

1. **True Positive** — correctly identified subscriber.
2. **False Positive** — predicted subscriber who did not subscribe.
3. **False Negative** — actual subscriber missed by the model.

For reproducibility, select the highest-confidence example available in each category.


In [ ]:
predictions["error_type"] = np.select(
    [
        (predictions.y_true == 0) & (predictions.y_pred == 0),
        (predictions.y_true == 0) & (predictions.y_pred == 1),
        (predictions.y_true == 1) & (predictions.y_pred == 0),
        (predictions.y_true == 1) & (predictions.y_pred == 1),
    ],
    ["TN", "FP", "FN", "TP"],
    default="UNKNOWN"
)

predictions["confidence"] = np.where(
    predictions["predicted_probability"] >= 0.5,
    predictions["predicted_probability"],
    1 - predictions["predicted_probability"]
)

representative_cases = {}

for category in ["TP", "FP", "FN"]:
    subset = predictions[
        predictions["error_type"] == category
    ].sort_values(
        "confidence",
        ascending=False
    )

    if len(subset) == 0:
        representative_cases[category] = None
    else:
        row = subset.iloc[0]
        representative_cases[category] = {
            "customer_index": int(row["customer_index"]),
            "y_true": int(row["y_true"]),
            "y_pred": int(row["y_pred"]),
            "predicted_probability": float(
                row["predicted_probability"]
            ),
            "confidence": float(row["confidence"]),
        }

print(json.dumps(
    representative_cases,
    indent=2
))

with open(CASE_PATH, "w", encoding="utf-8") as f:
    json.dump(
        representative_cases,
        f,
        indent=2
    )


# 6. Configure PyTorch Geometric GNNExplainer

The explanation is performed for the **customer node**.

The explainer learns masks over:

- node features
- graph edges

while keeping the trained GraphSAGE model fixed.

The explanation is therefore local to a specific customer prediction.


In [ ]:
explainer = Explainer(
    model=model,
    algorithm=GNNExplainer(
        epochs=100,
        lr=0.01
    ),
    explanation_type="model",
    node_mask_type="attributes",
    edge_mask_type="object",
    model_config=dict(
        mode="binary_classification",
        task_level="node",
        return_type="raw",
    ),
)

print("GNNExplainer configured.")


# 7. Explain One Customer

This helper generates a local explanation for a selected customer node.

The model itself is never updated.


In [ ]:
def explain_customer(customer_index):
    customer_index = int(customer_index)

    explanation = explainer(
        x=data.x_dict,
        edge_index=data.edge_index_dict,
        index=customer_index
    )

    return explanation


available_cases = {
    k: v for k, v in representative_cases.items()
    if v is not None
}

if not available_cases:
    raise RuntimeError(
        "No TP, FP, or FN test cases were available."
    )

print(
    "Cases available for explanation:",
    list(available_cases.keys())
)


# 8. Generate Representative Explanations

Generate explanations for TP, FP and FN where available.

Depending on graph size and hardware, GNNExplainer can take some time because it optimizes a local explanation mask for each case.


In [ ]:
explanations = {}

for category, case in available_cases.items():
    print("=" * 80)
    print("Explaining:", category)
    print("Customer index:", case["customer_index"])
    print("Probability:", case["predicted_probability"])

    explanations[category] = explain_customer(
        case["customer_index"]
    )

    print("Explanation generated.")


# 9. Inspect Explanation Structure

Check that the explanation contains feature masks and edge masks for the heterogeneous graph.


In [ ]:
for category, explanation in explanations.items():
    print("=" * 80)
    print("CASE:", category)
    print("Node masks:", getattr(explanation, "node_mask", None))
    print("Edge masks:", getattr(explanation, "edge_mask", None))
    print("Available explanation node types:", explanation.node_mask_dict.keys())
    print("Available explanation edge types:", explanation.edge_mask_dict.keys())


# 10. Customer Feature Importance

For the customer node, aggregate the learned attribute mask across feature dimensions.

Higher values indicate features that were more important to the local prediction according to GNNExplainer.

These are **model-attribution scores**, not causal effects.


In [ ]:
feature_rows = []

customer_feature_names = [
    f"customer_feature_{i}"
    for i in range(
        data["customer"].x.shape[1]
    )
]

for category, explanation in explanations.items():
    customer_mask = explanation.node_mask_dict.get(
        "customer"
    )

    if customer_mask is None:
        continue

    mask = (
        customer_mask
        .detach()
        .cpu()
    )

    # For node-level explanations the mask may contain
    # all nodes × features. Select the explained customer.
    if mask.dim() == 2:
        local_mask = mask[available_cases[category]["customer_index"]]
    else:
        local_mask = mask

    local_mask = local_mask.abs().numpy()

    for feature_index, importance in enumerate(local_mask):
        feature_rows.append({
            "case": category,
            "feature_index": feature_index,
            "feature_name": customer_feature_names[feature_index],
            "importance": float(importance),
        })

feature_importance = pd.DataFrame(
    feature_rows
)

if len(feature_importance):
    display(
        feature_importance
        .sort_values(
            ["case", "importance"],
            ascending=[True, False]
        )
        .groupby("case")
        .head(15)
    )
else:
    print("No customer feature mask was returned.")


# 11. Aggregate Feature Importance Across Cases

Compare the local explanations across TP, FP and FN.

A feature that repeatedly appears near the top is a useful candidate for deeper interpretation.


In [ ]:
if len(feature_importance):
    aggregate_feature_importance = (
        feature_importance
        .groupby(
            ["feature_index", "feature_name"],
            as_index=False
        )["importance"]
        .mean()
        .sort_values(
            "importance",
            ascending=False
        )
    )

    display(
        aggregate_feature_importance.head(20)
    )

    aggregate_feature_importance.to_csv(
        FEATURE_IMPORTANCE_PATH,
        index=False
    )
else:
    aggregate_feature_importance = pd.DataFrame(
        columns=[
            "feature_index",
            "feature_name",
            "importance"
        ]
    )


# 12. Visualize Feature Importance

Plot the strongest customer-node feature attributions across the representative cases.


In [ ]:
if len(feature_importance):
    top_features = (
        aggregate_feature_importance
        .head(15)
        .sort_values("importance")
    )

    plt.figure(figsize=(10, 7))
    plt.barh(
        top_features["feature_name"],
        top_features["importance"]
    )
    plt.xlabel("Mean GNNExplainer attribution")
    plt.ylabel("Customer feature")
    plt.title(
        "Top Customer Features in Representative Explanations"
    )
    plt.tight_layout()
    plt.savefig(
        PLOT_PATH,
        dpi=150,
        bbox_inches="tight"
    )
    plt.show()
else:
    print("Feature-importance plot skipped.")


# 13. Heterogeneous Relation / Edge Importance

GraphSAGE receives information through heterogeneous relations.

For each representative customer, summarize the total edge-mask importance by relation type.

This answers questions such as:

- Did `customer → job` contribute strongly?
- Did `customer → education` contribute?
- Did `customer → contact` contribute?
- Did `customer → campaign_month` contribute?

The exact relation names come from the graph itself.


In [ ]:
relation_rows = []

for category, explanation in explanations.items():
    for edge_type, mask in explanation.edge_mask_dict.items():
        if mask is None:
            continue

        edge_importance = (
            mask.detach()
            .abs()
            .cpu()
            .numpy()
        )

        relation_rows.append({
            "case": category,
            "source": edge_type[0],
            "relation": edge_type[1],
            "target": edge_type[2],
            "mean_edge_importance": float(
                edge_importance.mean()
            ),
            "max_edge_importance": float(
                edge_importance.max()
            ),
            "edge_count": int(
                edge_importance.size
            ),
        })

relation_importance = pd.DataFrame(
    relation_rows
)

if len(relation_importance):
    display(
        relation_importance.sort_values(
            "mean_edge_importance",
            ascending=False
        )
    )

    relation_importance.to_csv(
        RELATION_IMPORTANCE_PATH,
        index=False
    )
else:
    print("No edge masks were returned.")


# 14. Inspect the Most Important Edges

For each representative case, identify the strongest individual graph edges.

These edge masks are local explanations and should be interpreted as model reliance, not causal relationships.


In [ ]:
important_edges = []

for category, explanation in explanations.items():
    for edge_type, mask in explanation.edge_mask_dict.items():
        if mask is None:
            continue

        values = (
            mask.detach()
            .abs()
            .cpu()
            .numpy()
        )

        if len(values) == 0:
            continue

        top_k = min(10, len(values))
        top_indices = np.argsort(values)[-top_k:][::-1]

        edge_index = data[edge_type].edge_index.cpu().numpy()

        for position in top_indices:
            important_edges.append({
                "case": category,
                "source_type": edge_type[0],
                "relation": edge_type[1],
                "target_type": edge_type[2],
                "source_index": int(edge_index[0, position]),
                "target_index": int(edge_index[1, position]),
                "importance": float(values[position]),
            })

important_edges_df = pd.DataFrame(
    important_edges
)

if len(important_edges_df):
    display(
        important_edges_df.head(50)
    )
else:
    print("No individual edge explanations available.")


# 15. Save Explanation Summary

Save the local explanations in compact tabular form for later use by the backend.

The backend can later expose these explanations for a selected customer.


In [ ]:
summary_rows = []

for category, case in available_cases.items():
    summary_rows.append({
        "case": category,
        "customer_index": case["customer_index"],
        "y_true": case["y_true"],
        "y_pred": case["y_pred"],
        "predicted_probability": case["predicted_probability"],
        "confidence": case["confidence"],
    })

explanation_summary = pd.DataFrame(
    summary_rows
)

explanation_summary.to_csv(
    EXPLANATION_SUMMARY_PATH,
    index=False
)

print("Saved:", EXPLANATION_SUMMARY_PATH)
print("Saved:", CASE_PATH)

if len(feature_importance):
    print("Saved:", FEATURE_IMPORTANCE_PATH)

if len(relation_importance):
    print("Saved:", RELATION_IMPORTANCE_PATH)

print("Saved:", PLOT_PATH)


# 16. Explainability Interpretation Rules

The following rules should be preserved when this analysis is used in the application:

### 1. Attribution is not causation

A high GNNExplainer importance means the model relied on that feature/relation for the local prediction.

It does **not** mean the feature caused the customer to subscribe.

### 2. Local explanations are customer-specific

The explanation for one customer should not automatically be presented as the explanation for every customer.

### 3. Edge importance is graph-relative

An important edge means the model relied on that graph connection during message passing.

### 4. Do not convert attribution into unsupported business claims

For example:

> "Contact type caused subscription"

would be inappropriate.

A defensible statement is:

> "The model placed relatively high importance on the contact-related representation for this prediction."

### 5. The final model remains frozen

Explainability does not trigger another tuning cycle.


# 17. Final Verification

Notebook 08 passes when:

- the frozen final model loads
- final predictions are reproduced
- representative TP/FP/FN cases are identified
- GNNExplainer runs
- customer feature masks are extracted where supported
- heterogeneous edge/relation masks are extracted where supported
- explanation artifacts are saved
- the model remains frozen

### STOP POINT

At this point the complete ML/research phase is finished.

Next stage:

**FastAPI backend implementation**

The backend will load:

```text
hetero_graphsage_final.pt
```

and expose prediction + explanation functionality without retraining the model.


In [ ]:
required_outputs = [
    EXPLANATION_SUMMARY_PATH,
    CASE_PATH,
]

for path in required_outputs:
    assert path.exists(), f"Missing output: {path}"

assert checkpoint["freeze_status"] == "FINAL_FROZEN_MODEL"
assert config["hidden_dim"] == 128
assert abs(config["dropout"] - 0.20) < 1e-12

assert max_difference < 1e-7

print("=" * 90)
print("NOTEBOOK 08 VERIFICATION PASSED")
print("=" * 90)
print("Model status: FINAL_FROZEN_MODEL")
print("Representative cases:", list(available_cases.keys()))
print("Feature explanations:", len(feature_importance))
print("Relation explanations:", len(relation_importance))
print("Final model was not modified.")
print("=" * 90)
print("STOP: ML/research phase complete.")
print("NEXT: FastAPI backend.")
